<a href="https://colab.research.google.com/github/miizuha/miizuha.github.io/blob/main/2627-1/notebooks/lecture-02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bài 2 · Python cơ bản cho xử lý dữ liệu

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau notebook này, bạn:

1. Dùng list / dict / set đúng việc; phân biệt `None` với `0` và `""`.
2. Viết comprehension để lọc + biến đổi cả dãy dữ liệu.
3. Viết **hàm làm sạch chịu được dữ liệu bẩn** (try/except đúng loại lỗi).
4. Đọc/ghi CSV và JSON bằng Python thuần — hiểu những thứ mà pandas sẽ làm hộ từ buổi 4.

## 1. Dữ liệu demo: 8 phòng "kiểu Airbnb"

Một bảng dữ liệu, phiên bản Python thuần = **list các dict**. Đây là dữ liệu bẩn *có chủ đích*.

In [1]:
phong = [
    {"id": 101, "loai": "Entire home", "gia": "$1,200.00", "dem_toi_thieu": "2"},
    {"id": 102, "loai": "Private room", "gia": "$350.00",  "dem_toi_thieu": "1"},
    {"id": 103, "loai": "Entire home", "gia": "N/A",       "dem_toi_thieu": "3"},
    {"id": 104, "loai": "Private room", "gia": "$480.00",  "dem_toi_thieu": "1"},
    {"id": 105, "loai": "Entire home", "gia": "$2,150.00", "dem_toi_thieu": "999"},
    {"id": 106, "loai": "Shared room", "gia": "",          "dem_toi_thieu": "1"},
    {"id": 107, "loai": "Entire home", "gia": "$0.00",     "dem_toi_thieu": "2"},
    {"id": 108, "loai": "Private room", "gia": "$610.00",  "dem_toi_thieu": "1"},
]
len(phong), phong[0]

(8,
 {'id': 101, 'loai': 'Entire home', 'gia': '$1,200.00', 'dem_toi_thieu': '2'})

Để ý: giá là **chuỗi** (`"$1,200.00"`), có `"N/A"`, có rỗng, có `$0.00`, có phòng đòi ở 999 đêm.

## 2. list / dict / set — mỗi kiểu giải quyết một việc khác nhau

In [2]:
# dict: tra cứu theo khoá; .get() cho phương án B khi thiếu khoá
p = phong[0]
print(p["loai"])
print(p.get("diem_review", "chưa có dữ liệu"))

Entire home
chưa có dữ liệu


In [3]:
# slicing trên list: 3 phòng đầu, phòng cuối
print([p["id"] for p in phong[:3]])
print(phong[-1]["id"])

[101, 102, 103]
108


In [4]:
# set: so sánh hai snapshot — phòng nào biến mất, phòng nào mới xuất hiện?
snapshot_t9  = {101, 102, 103, 104, 105, 106}
snapshot_t12 = {101, 103, 105, 107, 108}

print("Biến mất :", snapshot_t9 - snapshot_t12)
print("Mới thêm :", snapshot_t12 - snapshot_t9)
print("Còn cả 2 :", snapshot_t9 & snapshot_t12)

Biến mất : {104, 106, 102}
Mới thêm : {107, 108}
Còn cả 2 : {105, 101, 103}


## 3. Comprehension: lọc + biến đổi cả dãy

So sánh hai cách viết cho cùng một công việc — lấy danh sách loại phòng, không trùng lặp:

In [5]:
# Cách 1: vòng for cổ điển
cac_loai = []
for p in phong:
    if p["loai"] not in cac_loai:
        cac_loai.append(p["loai"])
print(cac_loai)

# Cách 2: một dòng, rõ ý hơn
print(sorted({p["loai"] for p in phong}))     # set comprehension + sắp xếp

['Entire home', 'Private room', 'Shared room']
['Entire home', 'Private room', 'Shared room']


In [6]:
# dict comprehension: ánh xạ id -> loại phòng
id_toi_loai = {p["id"]: p["loai"] for p in phong}
id_toi_loai

{101: 'Entire home',
 102: 'Private room',
 103: 'Entire home',
 104: 'Private room',
 105: 'Entire home',
 106: 'Shared room',
 107: 'Entire home',
 108: 'Private room'}

In [7]:
# ⚠️ Gán KHÔNG phải sao chép
goc = [1, 2, 3]
alias = goc          # chỉ là tên khác của cùng một list
alias.append(99)
print("goc =", goc, "  <-- bị sửa theo!")

ban_sao = goc.copy()  # đây mới là bản sao
ban_sao.append(-1)
print("goc =", goc, "| ban_sao =", ban_sao)

goc = [1, 2, 3, 99]   <-- bị sửa theo!
goc = [1, 2, 3, 99] | ban_sao = [1, 2, 3, 99, -1]


## 4. Hàm làm sạch — chịu được dữ liệu bẩn

Cột `gia` đang là chuỗi kiểu `"$1,200.00"`. Muốn tính toán phải đổi sang số — và phải sống sót
qua `"N/A"`, chuỗi rỗng…

In [13]:
def clean_price(s):
    """Đổi chuỗi giá '$1,200.00' -> 1200.0; giá trị hỏng -> None."""
    try:
        return float(s.replace("$", "").replace(",", ""))
    except (ValueError, AttributeError):
        return None

# Bộ ca thử — luyện tập thói quen kiểm chứng trong môn này:
tests = ["$1,200.00", "N/A", "", "$0.00", None, "$2,150.00"]
for t in tests:
    print(f"{t!r:15} -> {clean_price(t)}")

'$1,200.00'     -> 1200.0
'N/A'           -> None
''              -> None
'$0.00'         -> 0.0
None            -> None
'$2,150.00'     -> 2150.0


In [14]:
# Áp hàm cho cả bảng, thêm khoá mới 'gia_so'
for p in phong:
    p["gia_so"] = clean_price(p["gia"])

# Giá trung bình của các phòng CÓ giá hợp lệ (> 0)
gia_hop_le = [p["gia_so"] for p in phong if p["gia_so"]]   # None và 0 đều bị loại
print("Số phòng có giá hợp lệ:", len(gia_hop_le))
print(f"Giá trung bình: ${sum(gia_hop_le) / len(gia_hop_le):,.2f}")

Số phòng có giá hợp lệ: 5
Giá trung bình: $958.00


Câu hỏi tự học: dòng `if p["gia_so"]` loại cả `None` **lẫn** `0.0` —
ta có *muốn* loại phòng giá $0 không? Đó là **quyết định QA**, phải ghi lại lý do.

## 5. Đọc & ghi file: CSV và JSON bằng tay (một lần để hiểu)

In [8]:
import csv
from pathlib import Path

# Ghi bảng ra CSV
Path("data").mkdir(exist_ok=True)
with open("data/phong.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["id", "loai", "gia"])
    w.writeheader()
    for p in phong:
        w.writerow({"id": p["id"], "loai": p["loai"], "gia": p["gia"]})

print(Path("data/phong.csv").read_text(encoding="utf-8")[:120], "…")

id,loai,gia
101,Entire home,"$1,200.00"
102,Private room,$350.00
103,Entire home,N/A
104,Private room,$480.00
105,Entire …


In [9]:
# Đọc lại: CSV không mang kiểu dữ liệu, mọi thứ thành chuỗi
with open("data/phong.csv", encoding="utf-8") as f:
    doc_lai = list(csv.DictReader(f))

doc_lai[0]          # id giờ là chuỗi '101'!

{'id': '101', 'loai': 'Entire home', 'gia': '$1,200.00'}

In [15]:
import json

# JSON thì giữ kiểu: số là số, None là null
tong_hop = {
    "nguon": "demo bài 2",
    "so_phong": len(phong),
    "gia_trung_binh": round(sum(gia_hop_le) / len(gia_hop_le), 2),
}
with open("data/tong_hop.json", "w", encoding="utf-8") as f:
    json.dump(tong_hop, f, ensure_ascii=False, indent=2)

print(Path("data/tong_hop.json").read_text(encoding="utf-8"))

{
  "nguon": "demo bài 2",
  "so_phong": 8,
  "gia_trung_binh": 958.0
}


## 6. Bài tập tại lớp

### Bài 1 — Hàm `clean_nights`

Cột `dem_toi_thieu` đang là chuỗi. Viết hàm đổi nó sang `int`; giá trị không đổi được → `None`.
Sau đó chạy cell test bên dưới — **phải qua hết cả 5 ca**.

In [17]:
def clean_nights(s):
    # TODO: viết thân hàm (gợi ý: int() + try/except ValueError/TypeError)
    try:
        return int(s)
    except (ValueError, TypeError):
        return None

# Bộ test — không sửa cell này:
assert clean_nights("2") == 2
assert clean_nights("999") == 999
assert clean_nights("N/A") is None
assert clean_nights(None) is None
assert clean_nights("") is None
print("✅ clean_nights qua cả 5 ca thử!")

✅ clean_nights qua cả 5 ca thử!


### Bài 2 — Comprehension

Dùng **một** comprehension cho mỗi câu:

1. Danh sách `id` của các phòng loại `"Entire home"`.
2. Dict `id -> gia_so` nhưng **chỉ** gồm phòng có giá hợp lệ (khác None và > 0).

In [18]:
# TODO Bài 2 — thay các dòng dưới bằng comprehension của bạn:
cau_1 = [p["id"] for p in phong if p["loai"] == "Entire home"]
cau_2 = {p["id"]: p["gia_so"] for p in phong if p["gia_so"]}

print("Câu 1:", cau_1)
print("Câu 2:", cau_2)

Câu 1: [101, 103, 105, 107]
Câu 2: {101: 1200.0, 102: 350.0, 104: 480.0, 105: 2150.0, 108: 610.0}


### Bài 3 — Quy tắc QA đầu tiên của bạn

Phòng 105 đòi ở tối thiểu 999 đêm — gần như chắc chắn là "phòng ảo" (chủ nhà khoá lịch).
Viết đoạn code **gắn cờ** (thêm khoá `"nghi_van": True/False`) cho các phòng có
`dem_toi_thieu` > 365 **hoặc** giá không hợp lệ. In danh sách id bị gắn cờ.

*Lưu ý tinh thần buổi 10: gắn cờ, không xoá!*

In [19]:
# TODO Bài 3:
for p in phong:
    dem = clean_nights(p["dem_toi_thieu"])
    p["nghi_van"] = (dem is not None and dem > 365) or not p["gia_so"]

print("Phòng bị gắn cờ:", [p["id"] for p in phong if p["nghi_van"]])

Phòng bị gắn cờ: [103, 105, 106, 107]


## 7. Thử thách về nhà 🏆

**Mini-pipeline thuần Python** — để buổi 4 bạn *thấy* pandas tiết kiệm bao nhiêu thời gian chạy:

1. Viết hàm `load_csv(path)` đọc `data/phong.csv` thành list các dict.
2. Viết hàm `summarize(rows)` trả về dict: với **mỗi loại phòng** — số phòng, giá trung bình
   (bỏ giá hỏng/0), giá cao nhất.
3. Viết hàm `save_json(obj, path)` ghi kết quả ra `data/bao_cao.json`.
4. Nối 3 hàm thành pipeline một dòng: `save_json(summarize(load_csv(...)), ...)`.

Ràng buộc: chỉ dùng những gì có trong notebook này (csv, json, comprehension, try/except).
✅ Được dùng AI — nhớ quy trình 5 bước: tự phác → hỏi → **đọc hiểu** → kiểm chứng → khai báo.

In [ ]:
RUN_CHALLENGE = False   # đổi True khi làm — viết code của bạn bên dưới

if RUN_CHALLENGE:
    def load_csv(path):
        ...
    def summarize(rows):
        ...
    def save_json(obj, path):
        ...
    save_json(summarize(load_csv("data/phong.csv")), "data/bao_cao.json")

---

## Tóm tắt buổi học

| Ý chốt | Vì sao quan trọng |
|---|---|
| `None` ≠ `0` ≠ `""` | Nền của mọi xử lý missing values (buổi 10) |
| list = dãy, dict = tra cứu, set = so thành viên | Chọn đúng cấu trúc = code ngắn và đúng |
| Comprehension: biến đổi **cả dãy** | Tư duy vector hoá — NumPy tuần sau |
| Hàm nhỏ + try/except đúng loại lỗi | Đơn vị xây pipeline bài tập lớn |
| CSV không mang kiểu; JSON ↔ dict | Hiểu thứ pandas sắp làm hộ bạn |

**Buổi sau:** NumPy — khi dữ liệu là *hàng triệu* con số, vòng for sẽ lộ nhược điểm.